# 4. Evaluate the completed extension and state what the evidence supports

This notebook brings together individual prediction error, Becker-style equal-cost calibration,
paired uncertainty, subgroup screening and final limitations. It distinguishes a worse point
estimate, statistically inconclusive evidence, and identical predictions chosen by fallback.
Default results are the completed real-data experiment, not the bounded examples in prior notebooks.

## 4.1 Setup and provenance

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "extension/clustering/supervised/pipeline.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside the prediction-market-analysis repository.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "extension/notebooks/results/.matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(ROOT / "extension/notebooks/results/.cache"))

In [ ]:
import json

import numpy as np
import pandas as pd
from IPython.display import Image, display
from threadpoolctl import threadpool_limits

from extension.notebooks.support import (
    begin_stage,
    finish_stage,
    output_dir,
    published_paths,
    read_completed,
)

published, DATA, ASSIGNMENTS, STATS, RESULTS = published_paths(ROOT)
print("Completed experiment:", RESULTS.relative_to(ROOT))
print("CSV split design:", published.get("split", "See methodology"))

In [ ]:
RUN_NAME = "completion_walkthrough"
RECOMPUTE = False  # Explicit opt-in: expensive steps write isolated notebook artifacts.
USE_REBUILT_UPSTREAM = False  # True only after completing the preceding rebuild notebooks.

evaluation_output = output_dir(ROOT, RUN_NAME, "evaluation")
if USE_REBUILT_UPSTREAM:
    model_input = output_dir(ROOT, RUN_NAME, "models")
    prediction_manifest = read_completed(model_input, source=ROOT, stage="prediction_fit")
    assert prediction_manifest.get("predictions_completed") is True
else:
    model_input = RESULTS
print("Input predictions:", model_input / "heldout_predictions.parquet")
print("Optional evaluation output:", evaluation_output)

## 4.2 Optional: recompute evaluation without retraining
This calls the same evaluation and report functions as the CLI. No model is selected or fitted
on test outcomes. Copies preserve original model settings/support and keep reports/figures away
from canonical artifacts. Completion is set only after evaluation and report generation succeed.
A report is an evaluation product, not an additional independent experiment.

In [ ]:
from extension.clustering.supervised.pipeline import evaluate
from extension.clustering.supervised.report import write_report

if RECOMPUTE:
    import shutil

    input_manifest_path = model_input / "run_manifest.json"
    evaluation_manifest = begin_stage(
        ROOT,
        evaluation_output,
        "evaluation",
        [model_input / "heldout_predictions.parquet", input_manifest_path],
        {"bootstrap": 2000, "prediction_directory": str(model_input)},
    )
    # Copy fit artifacts; never overwrite the original predictions or reports.
    for source in model_input.iterdir():
        if source.is_file() and source.name not in {"run_manifest.json", "REPORT.md"}:
            shutil.copy2(source, evaluation_output / source.name)
    for name in ["README.md", "METHODOLOGY.md", "CHANGES.md"]:
        shutil.copy2(ROOT / "extension/clustering/supervised" / name, evaluation_output / name)
    full_held = pd.read_parquet(model_input / "heldout_predictions.parquet")
    with threadpool_limits(limits=2):
        evaluate(full_held, evaluation_output, bootstrap=2000)
    write_report(evaluation_output, evaluation_output / "REPORT.md")
    finish_stage(
        evaluation_output,
        evaluation_manifest,
        evaluation_completed=True,
        note="Isolated notebook evaluation; original pipeline manifest retained as an input.",
    )
    active_results = evaluation_output
else:
    active_results = RESULTS

## 4.3 Population and individual prediction scores
Brier is execution-weighted squared error between predicted YES probability q and settlement y.
It also equals squared error of predicted residual versus y-p/100. Residual RMSE is 100*sqrt(Brier)
and residual MAE is 100*mean(abs(y-q)), in probability points. MAE describes noisy realized errors;
it is not a proper binary probability scoring rule, so validation selects using Brier.
Unit- and contract-balanced Brier are evaluation sensitivities, not separate replications.

In [ ]:
metrics = pd.read_csv(active_results / "metrics.csv")
coverage = pd.read_csv(active_results / "holdout_coverage.csv")
display(coverage)
assert np.allclose(metrics.residual_rmse_pp, 100 * np.sqrt(metrics.brier))
names = [
    "market_price",
    "historical_pooled",
    "historical_family",
    "historical_cluster",
    "pooled_ridge",
    "family_ridge",
    "cluster_ridge",
    "selected_pooled",
    "selected_family",
    "selected_cluster",
]
test_scores = metrics[metrics.split.eq("test")].set_index("model")
display(test_scores.loc[names, ["brier", "improvement_vs_price", "residual_mae_pp", "residual_rmse_pp"]])

## 4.4 Compare identical observations with paired event/series uncertainty
Positive improvement means the candidate has lower squared error than its baseline. The default
run bootstraps complete test events 2,000 times, pairing all repeated observations across models.
Series blocking is a coarser sensitivity; neither key fully recovers all shared outcomes.
Models, selected settings and observed covariates/support are held fixed. These intervals omit
training, clustering and model-selection uncertainty. Holm correction includes all 25 aggregate
contrasts, separately by split/block definition, including legacy comparisons.
A nominal p-value below .05 or a nominal interval excluding zero is not the full significance test.

In [ ]:
comparisons = pd.read_csv(active_results / "comparisons.csv")
test_comparisons = comparisons[comparisons.split.eq("test")]
historical = test_comparisons.candidate.str.startswith("historical_") | test_comparisons.baseline.str.startswith(
    "historical_"
)
columns = [
    "candidate",
    "baseline",
    "dependence_unit",
    "improvement",
    "ci_low",
    "ci_high",
    "p_value",
    "holm_p",
    "blocks",
]
display(test_comparisons.loc[historical, columns])
display(
    test_comparisons[test_comparisons.candidate.eq("selected_cluster") & test_comparisons.baseline.eq("market_price")][
        columns
    ]
)

## 4.5 Group calibration: errors by direction and the equal-cost wedge
At each cost c, compare YES with NO purchased at c (the NO position comes from YES price 100-c).
Both sides need 5 contracts and 5 events per cost; retain 1-49 cents and at least 10 bilateral
costs per cluster/split. Average each direction's error equally over those exact shared costs,
then subtract NO from YES for the wedge. The upper half would duplicate this both-side contrast.
Historical training bands do not replace exact-cost evaluation. Support can vary across groups/splits.

These scores weight supported NLP clusters equally, unlike execution-weighted Brier. Negative
wedge means relative YES underperformance, not necessarily absolute YES overpricing. Keep signed
YES/NO errors visible; no wedge-accuracy confidence intervals or significant wedge discoveries
are claimed. Sign agreement is descriptive, including uncertain near-zero realized wedges.

In [ ]:
wedge_metrics = pd.read_csv(active_results / "wedge_metrics.csv")
display(
    wedge_metrics[wedge_metrics.split.eq("test")]
    .set_index("model")
    .loc[names, ["clusters", "wedge_mae_pp", "wedge_rmse_pp", "yes_mae_pp", "no_mae_pp", "sign_agreement"]]
)
wedges = pd.read_csv(active_results / "cluster_wedges.csv")
# Ordered by group name, not by successful prediction: illustration only.
display(
    wedges[wedges.split.eq("test") & wedges.model.eq("historical_cluster")]
    .sort_values("cluster")[
        [
            "cluster",
            "costs",
            "observed_yes_pp",
            "observed_no_pp",
            "predicted_yes_pp",
            "predicted_no_pp",
            "observed_wedge_pp",
            "predicted_wedge_pp",
        ]
    ]
    .head(10)
)

## 4.6 Exploratory within-cluster comparisons
The full table includes every supported cluster rather than selecting successful examples.
Inference requires 30 held-out events; other comparisons receive p=1 for multiplicity correction.
Holm covers all ten contrasts times all supported clusters per split. Event concentration,
shared outcome drivers, prior exploration and normal-approximation limits still matter.
Do not treat several contracts from one ceremony/game as independent underlying outcomes.

In [ ]:
local = pd.read_csv(active_results / "cluster_comparisons.csv")
local_test = local[local.split.eq("test")]
summary = local_test.groupby(["candidate", "baseline"]).apply(
    lambda g: pd.Series(
        {
            "clusters": len(g),
            "eligible": int(g.inference_eligible.sum()),
            "holm_positive": int((g.holm_p.lt(0.05) & g.improvement.gt(0)).sum()),
            "holm_negative": int((g.holm_p.lt(0.05) & g.improvement.lt(0)).sum()),
        }
    ),
    include_groups=False,
)
display(summary)

## 4.7 Separate the three possible interpretations
**Worse point estimate:** the unshrunk cluster ridge had higher test Brier than pooled ridge.
Its nominal event-block degradation did not survive all broader checks; do not claim a general
proof that clustering is harmful. NLP ridge and structural-family ridge were essentially tied.

**Inconclusive improvement:** validation-selected cluster shrinkage had a tiny test improvement
against price, with an interval spanning zero. Beating a weak learned pooled comparator alone
is not a useful pricing advantage.

**Identical selected predictions:** the historical hierarchy selected `inherit` at every level,
so its final predictions equal price. This is not a failed significance test establishing equality
of underlying biases; validation chose no adjustment within the evaluated candidate grid.
The assertions below verify the documented run rather than turning synthetic examples into evidence.
For a new dataset/specification these findings must be rewritten, not imposed on the results.

In [ ]:
historical_selected = json.loads((active_results / "historical_selected.json").read_text())
all_inherit = all(row["strength"] is None for row in historical_selected.values())
print("Historical scopes all selected parent fallback:", all_inherit)
if all_inherit:
    for name in ["historical_pooled", "historical_family", "historical_cluster"]:
        np.testing.assert_allclose(test_scores.loc[name, "brier"], test_scores.loc["market_price", "brier"], atol=1e-14)
    print("Selected historical baselines match market-price Brier; no distinct correction was selected.")
print(
    "Cluster ridge minus pooled ridge Brier:",
    test_scores.loc["cluster_ridge", "brier"] - test_scores.loc["pooled_ridge", "brier"],
)
image = active_results / "prediction_performance.png"
if image.exists():
    display(Image(filename=str(image)))

## 4.8 Final manuscript boundaries and reproducibility
Use wording such as: *We tested whether historical calibration among comparable contracts
improves retrospective held-out predictions; the tested grouping and correction procedures did
not establish reliable improvement over market prices.* This is conditional on the implemented
models/features/grid and existing extract, not a statement that mispricing is impossible to predict.

There is no fixed 72-hour forecast. Event-hash separation does not establish chronological label
availability, metadata versions or independence of all outcome drivers. The test extract was
inspected during development and revisions are documented: it is an explored retrospective holdout,
not fresh confirmatory evidence. No fees, executable quotes or fills are modeled, so probability
points are not trading returns. A later unexamined, properly timed sample would be needed for
prospective claims.

[Research report](../clustering/supervised/REPORT.md) ·
[methodology](../clustering/supervised/METHODOLOGY.md) ·
[revision log](../clustering/supervised/CHANGES.md).
Review-mode execution verifies notebook readability and artifact consistency; it does not
independently rerun all expensive analyses. The optional rebuild route and parity checks are
explained in [README.md](README.md).